# Exp1A Calibration: Corrected Marginals with Condition-Specific Baselines

**Not formal pass/fail.** Verifies the implementation before committing GPU budget.

- Probe: Llama base (unsloth/Meta-Llama-3.1-8B)
- Corpora: Chinese Wiki + Buckeye
- Conditions: intact, D0, D3 (M=50), D4
- Shuffles: 10
- Stores both PPL and NLL marginals

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_calibration')
BASE.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'buckeye': DATA / 'buckeye_processed/speaker_concatenated.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
M = 50
N_SHUFFLES = 10
SEED = 20260429

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Calibration: {N_SHUFFLES} shuffles, {list(CORPORA.keys())}')
print('Setup done')

In [ ]:
# === Load model ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Model loaded')

In [ ]:
# === Core functions ===

def split_context(ctx, M):
    return ctx[:len(ctx)-M], ctx[len(ctx)-M:]

def d0_noop(ctx, M=50):
    far, near = split_context(ctx, M)
    return far + near

def d3_swap(ctx, M=50):
    far, near = split_context(ctx, M)
    return near + far

def d4_reverse(ctx):
    return list(reversed(ctx))

@torch.no_grad()
def compute_target_ppl_nll(context_tokens, target_tokens):
    """Returns (ppl, mean_nll) for target given context."""
    if len(target_tokens) < 2:
        return float('inf'), float('inf')
    full_ids = list(context_tokens) + list(target_tokens)
    target_start = len(context_tokens)
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_nll = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_nll += -log_probs[full_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    if count == 0:
        return float('inf'), float('inf')
    mean_nll = total_nll / count
    return math.exp(mean_nll), mean_nll

def compute_condition_curves(condition_ctx, target_tokens, n_shuffles=N_SHUFFLES, seed=SEED):
    """Full corrected-marginal computation with condition-specific baselines."""
    max_c = len(condition_ctx)
    ord_ppl, ord_nll = [], []
    shuf_ppl_mean, shuf_ppl_sd = [], []
    shuf_nll_mean, shuf_nll_sd = [], []
    
    for c in range(max_c + 1):
        # Ordered
        if c == 0:
            ppl, nll = compute_target_ppl_nll([], target_tokens)
        else:
            prefix = condition_ctx[-c:]
            ppl, nll = compute_target_ppl_nll(prefix, target_tokens)
        ord_ppl.append(ppl)
        ord_nll.append(nll)
        
        # Shuffled baseline (condition-specific)
        if c == 0:
            shuf_ppl_mean.append(ppl)
            shuf_ppl_sd.append(0.0)
            shuf_nll_mean.append(nll)
            shuf_nll_sd.append(0.0)
        else:
            revealed = condition_ctx[-c:]
            rng = random.Random(seed + c)
            s_ppls, s_nlls = [], []
            for _ in range(n_shuffles):
                shuffled = list(revealed)
                rng.shuffle(shuffled)
                sp, sn = compute_target_ppl_nll(shuffled, target_tokens)
                if not math.isinf(sp):
                    s_ppls.append(sp)
                    s_nlls.append(sn)
            if s_ppls:
                shuf_ppl_mean.append(float(np.mean(s_ppls)))
                shuf_ppl_sd.append(float(np.std(s_ppls)))
                shuf_nll_mean.append(float(np.mean(s_nlls)))
                shuf_nll_sd.append(float(np.std(s_nlls)))
            else:
                shuf_ppl_mean.append(ppl)
                shuf_ppl_sd.append(0.0)
                shuf_nll_mean.append(nll)
                shuf_nll_sd.append(0.0)
    
    # Marginals
    distances = list(range(1, max_c + 1))
    m_ord_ppl = [ord_ppl[d-1] - ord_ppl[d] for d in distances]
    m_shuf_ppl = [shuf_ppl_mean[d-1] - shuf_ppl_mean[d] for d in distances]
    delta_ppl = [mo - ms for mo, ms in zip(m_ord_ppl, m_shuf_ppl)]
    m_ord_nll = [ord_nll[d-1] - ord_nll[d] for d in distances]
    m_shuf_nll = [shuf_nll_mean[d-1] - shuf_nll_mean[d] for d in distances]
    delta_nll = [mo - ms for mo, ms in zip(m_ord_nll, m_shuf_nll)]
    
    return {
        'distances': distances,
        'ordered_ppl': ord_ppl,
        'ordered_nll': ord_nll,
        'shuffled_ppl_mean': shuf_ppl_mean,
        'shuffled_ppl_sd': shuf_ppl_sd,
        'shuffled_nll_mean': shuf_nll_mean,
        'shuffled_nll_sd': shuf_nll_sd,
        'm_ordered_ppl': m_ord_ppl,
        'm_shuffled_ppl': m_shuf_ppl,
        'delta_ppl': delta_ppl,
        'm_ordered_nll': m_ord_nll,
        'm_shuffled_nll': m_shuf_nll,
        'delta_nll': delta_nll,
    }

print(f'Functions defined — {N_SHUFFLES} shuffles per c')

In [ ]:
# === D0 sanity check (2 docs, must match intact exactly for ordered) ===
print('='*60)
print('D0 SANITY CHECK')
print('='*60)

corpus = []
with open(CORPORA['wiki_zh']) as f:
    for i, line in enumerate(f):
        if i >= 2: break
        corpus.append(json.loads(line))

t0 = time.time()
for doc in corpus:
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    ts = int(n * 0.5)
    te = min(ts + TARGET_LEN, n)
    if ts < MIN_BEFORE: continue
    
    target = full_ids[ts:te]
    ctx = full_ids[ts-C:ts]
    
    print(f'  {doc["doc_id"][:30]}')
    
    # Intact ordered PPL
    intact_r = compute_condition_curves(ctx, target)
    # D0
    d0_ctx = d0_noop(ctx, M)
    d0_r = compute_condition_curves(d0_ctx, target)
    
    # Check ordered PPL match (should be exact)
    max_ppl_diff = max(abs(a-b) for a, b in zip(intact_r['ordered_ppl'], d0_r['ordered_ppl']))
    print(f'    Ordered PPL max diff: {max_ppl_diff:.8f} ({"PASS" if max_ppl_diff < 1e-6 else "FAIL"})')
    
    # Check corrected marginals (should differ only by Monte Carlo noise)
    delta_diff = [abs(a-b) for a, b in zip(intact_r['delta_ppl'], d0_r['delta_ppl'])]
    mean_delta_diff = np.mean(delta_diff)
    intact_mag = np.mean([abs(d) for d in intact_r['delta_ppl']])
    pct = mean_delta_diff / intact_mag * 100 if intact_mag > 0 else 0
    print(f'    Corrected marginal diff: {mean_delta_diff:.6f} ({pct:.1f}% of intact)')
    print(f'    Intact mean |delta|: {intact_mag:.6f}')

elapsed = time.time() - t0
secs_per_target = elapsed / 2 / 2  # 2 docs, 2 conditions each
print(f'\n  Time: {elapsed:.0f}s total, {secs_per_target:.0f}s per target-condition')
print(f'  With {N_SHUFFLES} shuffles: ~{secs_per_target * (N_SHUFFLES + 1) / (N_SHUFFLES + 1):.0f}s per target')

# Extrapolate
n_targets = sum(1 for _ in open(CORPORA['wiki_zh'])) * 3
n_conditions = 4  # intact, D0, D3, D4
est_hrs = secs_per_target * n_targets * n_conditions * len(CORPORA) / 3600
print(f'  Est. calibration run: ~{est_hrs:.1f} hours')

In [ ]:
# === Prefix sanity check ===
# Verify that revealed tokens at key positions match predictions

print('='*60)
print('PREFIX SANITY CHECK')
print('='*60)

doc = corpus[0]
full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
ts = int(len(full_ids) * 0.5)
ctx = full_ids[ts-C:ts]  # temporal order: oldest to newest

# Intact
intact_ctx = list(ctx)
d3_ctx = d3_swap(ctx, M)
d4_ctx = d4_reverse(ctx)

print(f'  Context length: {len(ctx)}')
print(f'  ctx[-1] (t_1, immediate predecessor): {ctx[-1]}')
print(f'  ctx[0]  (t_C, most distant):          {ctx[0]}')

print(f'\n  Intact c=1 reveals: {intact_ctx[-1:]}  (should be t_1 = {ctx[-1]})')
print(f'  D3 c=1 reveals:     {d3_ctx[-1:]}  (should be far boundary = {ctx[C-M-1]})')
print(f'  D4 c=1 reveals:     {d4_ctx[-1:]}  (should be t_C = {ctx[0]})')

print(f'\n  D3 c=51 reveals last 51 of d3_ctx:')
d3_at_51 = d3_ctx[-51:]
print(f'    First of these: {d3_at_51[0]} (should be original near-half token)')
print(f'    This is originally ctx[{C-M}] = {ctx[C-M]}: {d3_at_51[0] == ctx[C-M]}')

# Verify D3 shuffled baseline at c=1 uses the right token
print(f'\n  D3 shuffled at c=1: shuffles [{d3_ctx[-1]}] — single token, no permutation possible')
print(f'  D3 shuffled at c=60: shuffles {len(d3_ctx[-60:])} tokens from D3 prefix (NOT intact prefix)')

In [ ]:
# === Main calibration run ===
CONDITIONS = {
    'intact': lambda ctx: list(ctx),
    'D0': lambda ctx: d0_noop(ctx, M),
    'D3_M50': lambda ctx: d3_swap(ctx, M),
    'D4': lambda ctx: d4_reverse(ctx),
}

for corpus_name, corpus_path in CORPORA.items():
    print(f'\n{"="*60}')
    print(f'{corpus_name}')
    print(f'{"="*60}')
    
    docs = []
    with open(corpus_path) as f:
        for line in f:
            docs.append(json.loads(line))
    
    for cond_name, cond_fn in CONDITIONS.items():
        cache_path = BASE / f'llama_{corpus_name}_{cond_name}.json'
        if cache_path.exists():
            with open(cache_path) as f:
                cached = json.load(f)
            print(f'  {cond_name}: cached ({len(cached)} results)')
            continue
        
        print(f'  {cond_name}: processing...')
        t0 = time.time()
        results = []
        
        for doc in tqdm(docs, desc=f'{corpus_name}/{cond_name}'):
            full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(full_ids)
            
            for frac in TARGET_FRACS:
                ts = int(n * frac)
                te = min(ts + TARGET_LEN, n)
                if ts < MIN_BEFORE or te - ts < 5:
                    continue
                
                target = full_ids[ts:te]
                ctx = full_ids[ts-C:ts]
                disrupted = cond_fn(ctx)
                
                r = compute_condition_curves(disrupted, target)
                r['doc_id'] = doc.get('doc_id', '')
                r['target_frac'] = frac
                r['condition'] = cond_name
                results.append(r)
        
        elapsed = time.time() - t0
        with open(cache_path, 'w') as f:
            json.dump(results, f)
        print(f'    {len(results)} results in {elapsed/60:.1f} min')
        
        # Quick report
        if results:
            mean_delta = np.mean([np.mean(r['delta_ppl']) for r in results])
            print(f'    Mean corrected marginal (PPL): {mean_delta:.6f}')

print('\nCalibration run complete!')

In [ ]:
# === Analysis: D0, D3, D4 corrected marginals ===
from scipy.ndimage import uniform_filter1d
import matplotlib.pyplot as plt

for corpus_name in CORPORA:
    print(f'\n{"="*60}')
    print(f'{corpus_name} — Corrected Marginals')
    print(f'{"="*60}')
    
    data = {}
    for cond in ['intact', 'D0', 'D3_M50', 'D4']:
        cp = BASE / f'llama_{corpus_name}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        data[cond] = results
    
    if 'intact' not in data:
        print('  No intact data'); continue
    
    # === D0 check ===
    if 'D0' in data:
        intact_delta = np.mean([r['delta_ppl'] for r in data['intact']], axis=0)
        d0_delta = np.mean([r['delta_ppl'] for r in data['D0']], axis=0)
        diff = np.mean(np.abs(intact_delta - d0_delta))
        mag = np.mean(np.abs(intact_delta))
        pct = diff / mag * 100 if mag > 0 else 0
        n_ci_excl = 0  # simplified — full bootstrap needed for formal
        print(f'  D0: mean |diff| = {diff:.6f} ({pct:.1f}% of intact) — {"PASS" if pct < 5 else "CHECK"}')
    
    # === D3 jump ===
    if 'D3_M50' in data:
        d3_delta = np.mean([r['delta_ppl'] for r in data['D3_M50']], axis=0)
        jump = d3_delta[M] - d3_delta[M-1] if len(d3_delta) > M else 0
        pre_sd = np.std(d3_delta[:M])
        z = jump / pre_sd if pre_sd > 0 else 0
        print(f'  D3: jump at M+1 = {jump:.6f}, standardized = {z:.2f} (threshold: 0.5)')
    
    # === D4 reversal ===
    if 'D4' in data:
        intact_delta = np.mean([r['delta_ppl'] for r in data['intact']], axis=0)
        d4_delta = np.mean([r['delta_ppl'] for r in data['D4']], axis=0)
        rev_intact = intact_delta[::-1]
        rho_rev, p_rev = stats.spearmanr(d4_delta, rev_intact)
        rho_fwd, p_fwd = stats.spearmanr(d4_delta, intact_delta)
        contrast = rho_rev - rho_fwd
        print(f'  D4: rho(D4, rev_intact) = {rho_rev:.3f}, rho(D4, intact) = {rho_fwd:.3f}')
        print(f'      contrast = {contrast:.3f} (threshold: 0.3), rho_rev >= 0.6: {rho_rev >= 0.6}')
    
    # Also report magnitude effect
    if 'D4' in data:
        intact_ppl = np.mean([r['ordered_ppl'] for r in data['intact']], axis=0)
        d4_ppl = np.mean([r['ordered_ppl'] for r in data['D4']], axis=0)
        intact_benefit = intact_ppl[0] - intact_ppl[-1]
        d4_benefit = d4_ppl[0] - d4_ppl[-1]
        preserved = d4_benefit / intact_benefit if intact_benefit > 0 else 0
        print(f'      Intact PPL benefit: {intact_benefit:.1f}, D4 benefit: {d4_benefit:.1f} ({preserved:.0%} preserved)')

In [ ]:
# === Visualization ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, corpus_name in enumerate(CORPORA):
    # Corrected marginals
    ax = axes[idx, 0]
    for cond, color in [('intact','blue'), ('D3_M50','red'), ('D4','green')]:
        cp = BASE / f'llama_{corpus_name}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        smooth = uniform_filter1d(curve, 5)
        ax.plot(range(1, len(smooth)+1), smooth, color=color, linewidth=2, label=cond)
    ax.axvline(M, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(f'{corpus_name} — Corrected Marginals', fontweight='bold')
    ax.set_xlabel('Distance d')
    ax.set_ylabel('Δ_d (corrected marginal)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)
    
    # PPL curves
    ax = axes[idx, 1]
    for cond, color in [('intact','blue'), ('D3_M50','red'), ('D4','green')]:
        cp = BASE / f'llama_{corpus_name}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        ppl = np.mean([r['ordered_ppl'] for r in results], axis=0)
        ax.plot(range(len(ppl)), ppl, color=color, linewidth=2, label=cond)
    ax.axvline(M, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'{corpus_name} — Ordered PPL', fontweight='bold')
    ax.set_xlabel('Context length c')
    ax.set_ylabel('Perplexity')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

plt.suptitle('Exp1A Calibration: Corrected Marginals (10 shuffles)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved')

## Calibration Checklist

Before proceeding to formal run, verify:

- [ ] D0 ordered PPL matches intact exactly
- [ ] D0 corrected marginal within 5% of intact
- [ ] D3 shows visible jump in corrected marginals at M=50
- [ ] D4 corrected marginal curve differs from intact
- [ ] D4 Spearman contrast > 0.3 on corrected marginals
- [ ] No NaN/inf in corrected marginals
- [ ] Runtime is acceptable for full run
- [ ] Variance from 10 shuffles is reasonable (not dominated by MC noise)